<div align="center">
<img src="https://poorit.in/image.png" alt="Poorit" width="40" style="vertical-align: middle;"> <b>AI SYSTEMS ENGINEERING 2</b>

## Unit 4: LangGraph Fundamentals — State, Tools, and Memory

**CV Raman Global University, Bhubaneswar**
*AI Center of Excellence*

</div>

---

### What You'll Learn

In this notebook, you will:

1. **Understand LangGraph's core workflow** — the 5 steps: State → StateGraph → Nodes → Edges → Compile
2. **Build a three-node pipeline** — see that LangGraph is about Python functions, not just AI
3. **Use conditional edges** — let your graph make decisions and branch
4. **Add an LLM to a graph** — create a simple chatbot with ChatOpenAI
5. **Create and bind tools** — give your agent abilities with `Tool` and `bind_tools()`
6. **Use ToolNode and tools_condition** — let LangGraph handle tool execution automatically

## 1. Environment Setup

In [ ]:
!pip install -q langgraph langchain-openai langchain-community

In [ ]:
import os
import random
from typing import TypedDict
from langgraph.graph import StateGraph, START, END
from IPython.display import Image, display

In [ ]:
from getpass import getpass

api_key = getpass("Enter your OpenAI API Key: ")
os.environ['OPENAI_API_KEY'] = api_key
MODEL = "gpt-4o-mini"

---

## 2. What is LangGraph?

LangGraph is a framework for building **stateful, multi-step applications** with LLMs. It models your application as a **graph** — nodes do the work, edges control the flow.

| DIY Agent Loop | LangGraph |
|---|---|
| Manual state passing between steps | `State` class manages everything |
| Write your own routing logic | Conditional edges handle branching |
| Build your own tool loop | `ToolNode` + `tools_condition` built-in |
| No checkpointing | `MemorySaver` for conversation memory |

**The 5 Steps to Build a Graph:**
1. Define the **State** class
2. Create a **StateGraph** with that State
3. Add **Nodes** (Python functions)
4. Add **Edges** (connections)
5. **Compile** the graph

---

## 3. Your First Graph — A Three-Node Pipeline

Let's build a graph that chains **three nodes** into a pipeline. Each node reads from state, does its work, and writes back to state. The next node picks up where the previous one left off.

This demonstrates that LangGraph is fundamentally about **Python functions** — no LLM required.

**The 5 steps in action:**
1. **State** — `PipelineState` (a `TypedDict`) defines `topic`, `sentence`, and `formatted`
2. **StateGraph** — `StateGraph(PipelineState)` creates the graph builder
3. **Nodes** — three Python functions that read/write state
4. **Edges** — linear chain: `START → generate_topic → write_sentence → format_output → END`
5. **Compile** — `pipeline_builder.compile()` produces a runnable graph

```
START → generate_topic → write_sentence → format_output → END
```

In [ ]:
class PipelineState(TypedDict):
    topic: str
    sentence: str
    formatted: str

In [ ]:
topics = ["Penguins", "Volcanoes", "Robots", "Bananas", "Thunderstorms"]
templates = [
    "{topic} are more interesting than most people realize.",
    "Did you know? {topic} have a fascinating hidden side.",
    "The world would be very different without {topic}.",
]

def generate_topic(state: PipelineState):
    return {"topic": random.choice(topics)}

def write_sentence(state: PipelineState):
    template = random.choice(templates)
    return {"sentence": template.format(topic=state["topic"])}

def format_output(state: PipelineState):
    decorated = f"✨ {state['sentence'].upper()} ✨"
    return {"formatted": decorated}

In [ ]:
pipeline_builder = StateGraph(PipelineState)

# Add nodes
pipeline_builder.add_node("generate_topic", generate_topic)
pipeline_builder.add_node("write_sentence", write_sentence)
pipeline_builder.add_node("format_output", format_output)

# Add edges — linear chain
pipeline_builder.add_edge(START, "generate_topic")
pipeline_builder.add_edge("generate_topic", "write_sentence")
pipeline_builder.add_edge("write_sentence", "format_output")
pipeline_builder.add_edge("format_output", END)

pipeline_graph = pipeline_builder.compile()
display(Image(pipeline_graph.get_graph().draw_mermaid_png()))

In [ ]:
result = pipeline_graph.invoke({"topic": "", "sentence": "", "formatted": ""})
print(f"Topic:     {result['topic']}")
print(f"Sentence:  {result['sentence']}")
print(f"Formatted: {result['formatted']}")

> **Key insight:** Nodes are just Python functions, edges control the flow. Each node reads from and writes to the shared state.

---

## 4. Conditional Edges

So far our edges have been fixed — A always goes to B. But what if we want the graph to **make decisions**?

**Conditional edges** let a node route to different next nodes based on the current state. You provide a **router function** that returns the name of the next node.

Let's modify our pipeline: after `generate_topic`, if the topic is science-related, we write a sentence. Otherwise, we skip straight to formatting with a default message.

```
                              ┌── (science) ──→ write_sentence ──┐
START → generate_topic ──────┤                                    ├──→ format_output → END
                              └── (other) ─────────────────────────┘
```

In [ ]:
science_topics = ["Volcanoes", "Thunderstorms", "Robots"]

def route_by_topic(state: PipelineState) -> str:
    """Router function: returns the name of the next node."""
    if state["topic"] in science_topics:
        return "write_sentence"
    else:
        return "format_output"

def write_sentence_v2(state: PipelineState):
    template = random.choice(templates)
    return {"sentence": template.format(topic=state["topic"])}

def format_output_v2(state: PipelineState):
    sentence = state["sentence"] if state["sentence"] else f"{state['topic']}? Not science — skipped!"
    return {"formatted": f"✨ {sentence.upper()} ✨"}

In [ ]:
cond_builder = StateGraph(PipelineState)

cond_builder.add_node("generate_topic", generate_topic)
cond_builder.add_node("write_sentence", write_sentence_v2)
cond_builder.add_node("format_output", format_output_v2)

cond_builder.add_edge(START, "generate_topic")

# Conditional edge — router decides the next node
cond_builder.add_conditional_edges("generate_topic", route_by_topic)

cond_builder.add_edge("write_sentence", "format_output")
cond_builder.add_edge("format_output", END)

cond_graph = cond_builder.compile()
display(Image(cond_graph.get_graph().draw_mermaid_png()))

In [ ]:
# Run a few times — sometimes it picks a science topic (goes through write_sentence),
# sometimes not (skips to format_output)
for i in range(3):
    result = cond_graph.invoke({"topic": "", "sentence": "", "formatted": ""})
    print(f"Run {i+1}: topic={result['topic']:15s} → {result['formatted']}")

> **Key insight:** `add_conditional_edges(node, router_fn)` lets the graph branch. The router function inspects state and returns the name of the next node.

---

## 5. Adding an LLM

Now let's swap our random functions for a real LLM. When we use a chatbot, messages need to **accumulate** — each new message should be appended to the conversation, not replace it.

LangGraph handles this with a **reducer**. We annotate the `messages` field with `add_messages`, which tells LangGraph: *"when a node returns new messages, append them to the existing list instead of replacing it."*

```python
messages: Annotated[list, add_messages]  # append, don't replace
```

In [ ]:
from typing import Annotated
from langgraph.graph.message import add_messages
from langchain_openai import ChatOpenAI

In [ ]:
class ChatState(TypedDict):
    messages: Annotated[list, add_messages]

In [ ]:
llm = ChatOpenAI(model=MODEL)

def chatbot_node(state: ChatState):
    response = llm.invoke(state["messages"])
    return {"messages": [response]}

chat_builder = StateGraph(ChatState)
chat_builder.add_node("chatbot", chatbot_node)
chat_builder.add_edge(START, "chatbot")
chat_builder.add_edge("chatbot", END)

chat_graph = chat_builder.compile()
display(Image(chat_graph.get_graph().draw_mermaid_png()))

In [ ]:
result = chat_graph.invoke({"messages": [{"role": "user", "content": "What is the capital of India?"}]})
print(result["messages"][-1].content)

---

## 6. Tools

A chatbot that can only generate text is limited. **Tools** let your agent take actions — search, calculate, call APIs.

Adding tools to a LangGraph agent requires two things:
1. `llm.bind_tools(tools)` — tell the LLM what tools are available
2. `ToolNode(tools)` + `tools_condition` — handle tool execution and routing

```
         ┌──────────┐
START ──>│  chatbot  │──> tools_condition ──> END (if no tool call)
         └──────────┘         │
              ^               │ (if tool call)
              │               v
              │         ┌──────────┐
              └─────────│  tools   │
                        └──────────┘
```

### Creating Tools

LangChain's `Tool` class wraps a Python function so the LLM can call it:

In [ ]:
from langchain_core.tools import Tool
from langgraph.prebuilt import ToolNode, tools_condition

In [ ]:
def search(query: str) -> str:
    """Mock search function."""
    responses = {
        "capital": "Paris is the capital of France.",
        "weather": "The weather is sunny with temperatures around 25°C.",
    }
    for key, response in responses.items():
        if key in query.lower():
            return response
    return f"Search results for: {query}"

tool_search = Tool(
    name="search",
    func=search,
    description="Useful for searching information"
)

# Test it directly
tool_search.invoke("What is the capital of France?")

In [ ]:
def calculate(expression: str) -> str:
    """Evaluate a math expression."""
    try:
        return f"Result: {eval(expression)}"
    except Exception as e:
        return f"Error: {str(e)}"

tool_calculator = Tool(
    name="calculator",
    func=calculate,
    description="Useful for math calculations"
)

tools = [tool_search, tool_calculator]

### Building the Tool-Using Agent

In [ ]:
class ToolState(TypedDict):
    messages: Annotated[list, add_messages]

In [ ]:
llm_with_tools = ChatOpenAI(model=MODEL).bind_tools(tools)

def chatbot(state: ToolState):
    return {"messages": [llm_with_tools.invoke(state["messages"])]}

In [ ]:
graph_builder = StateGraph(ToolState)
graph_builder.add_node("chatbot", chatbot)
graph_builder.add_node("tools", ToolNode(tools=tools))

# Conditional edge: if LLM calls a tool, go to tools node; otherwise, END
graph_builder.add_conditional_edges("chatbot", tools_condition)
graph_builder.add_edge("tools", "chatbot")  # After tools, back to chatbot
graph_builder.add_edge(START, "chatbot")

tool_graph = graph_builder.compile()
display(Image(tool_graph.get_graph().draw_mermaid_png()))

In [ ]:
# Test: should use the search tool
result = tool_graph.invoke({"messages": [{"role": "user", "content": "What is the capital of France?"}]})
print(result["messages"][-1].content)

In [ ]:
# Test: should use the calculator tool
result = tool_graph.invoke({"messages": [{"role": "user", "content": "What is 25 * 17 + 3?"}]})
print(result["messages"][-1].content)

> **How it works:** `tools_condition` checks if the LLM's response contains tool calls. If yes → route to `ToolNode`. If no → route to `END`. The `ToolNode` executes the tool and sends the result back to the chatbot.

---

## 7. Key Takeaways

### Concept Map

```
State (TypedDict or Annotated)  ─── defines what the graph tracks
  │
  └── passed to ──> StateGraph(State)
                      │
                      ├── add_node("name", fn)          ─── Python function as a node
                      ├── add_node("tools", ToolNode)   ─── auto-executes tool calls
                      │
                      ├── add_edge(START, "name")       ─── fixed routing
                      ├── add_conditional_edges(         ─── dynamic routing
                      │     "node", router_fn)
                      │
                      └── compile()
                            │
                            └── graph.invoke(state)
```

### Quick Reference

| Concept | Code | Purpose |
|---|---|---|
| **State** | `class State(TypedDict)` | Define what data the graph tracks |
| **Reducer** | `Annotated[list, add_messages]` | How state updates are combined (append vs replace) |
| **Node** | `graph.add_node(name, fn)` | A processing step (Python function) |
| **Edge** | `graph.add_edge(a, b)` | Fixed connection between nodes |
| **Conditional edge** | `add_conditional_edges(node, fn)` | Dynamic routing based on state |
| **Bind tools** | `llm.bind_tools(tools)` | Tell the LLM about available tools |
| **ToolNode** | `ToolNode(tools=tools)` | Auto-execute tool calls |
| **tools_condition** | `tools_condition` | Route to tools or END |

---

## 8. Exercises

### Exercise 1: Graph with Conditional Edge (Beginner)
Build a 3-node graph (no LLM) that takes user input and routes conditionally. For example:
- Node 1: receives a number from state
- Conditional edge: if the number is even → Node 2 (doubles it), if odd → Node 3 (triples it and adds 1)
- Visualize the graph, invoke it with different inputs, and verify the routing works.

---

**What's Next?** In the next notebook, we'll explore memory with MemorySaver, async LangGraph, human-in-the-loop patterns, and persistent memory with SQLite.

---

**Course Information:**
- **Institution:** CV Raman Global University, Bhubaneswar
- **Program:** AI Center of Excellence
- **Course:** AI Systems Engineering 2
- **Developed by:** [Poorit Technologies](https://poorit.in) - *Transform Graduates into Industry-Ready Professionals*

---